# RAG Assignment - Development & Testing Notebook
This notebook tests each of the custom functions in `functions.py` individually,
as required by the assignment deliverables (Tasks 1, 2, 3, 5).

## Setup

In [1]:
!pip install -q langchain_text_splitters sentence-transformers scikit-learn PyPDF2 cohere


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from functions import (
    load_and_chunk_document,
    create_embeddings,
    search_chunks,
    generate_answer,
)
import numpy as np

c:\Users\Luv Mangla\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task 1: Custom Document Loader
`load_and_chunk_document(file_path, chunk_size=300, overlap=50)`

Test run on `sample_document.txt` (a short article about RAG systems).

In [3]:
chunks = load_and_chunk_document("sample_document.txt", chunk_size=300, overlap=50)

print(f"Number of chunks: {len(chunks)}")
print()
for i, c in enumerate(chunks):
    print(f"--- Chunk {i+1} (length={len(c)}) ---")
    print(c)
    print()

Number of chunks: 10

--- Chunk 1 (length=300) ---
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, RAG retrieves relevant documents from an external knowledge base and uses them as context to generate more accurate,

--- Chunk 2 (length=65) ---
uses them as context to generate more accurate, grounded answers.

--- Chunk 3 (length=297) ---
A typical RAG pipeline has three main stages. First, documents are split into smaller chunks so they can be processed efficiently and fit within context limits. Second, each chunk is converted into a numerical vector using an embedding model, and these vectors are stored in a vector database such

--- Chunk 4 (length=257) ---
vectors are stored in a vector database such as ChromaDB or FAISS. Third, when a user asks a question, the question itself is embedded, and the system searches the vector database for the most si

## Task 2: Custom Embedding Function
`create_embeddings(chunks, model_name="all-MiniLM-L6-v2")`

Test run showing the embedding shape.

> **Note:** The cells below download the `all-MiniLM-L6-v2` model from Hugging Face
> the first time they run, so an internet connection is required (this works fine on
> Google Colab or a local machine — just needs a few seconds to download on first run).

In [4]:
embeddings = create_embeddings(chunks)

print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimension (shape of each vector): {len(embeddings[0])}")
print(f"Type check: list of lists of floats -> {isinstance(embeddings, list) and isinstance(embeddings[0], list) and isinstance(embeddings[0][0], float)}")
print()
print("First 10 values of first embedding vector:")
print(embeddings[0][:10])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2469.92it/s]


Number of embeddings: 10
Embedding dimension (shape of each vector): 384
Type check: list of lists of floats -> True

First 10 values of first embedding vector:
[-0.07782413810491562, 0.05182896926999092, -0.029850441962480545, 0.036954957991838455, -0.05744262412190437, 0.04741981625556946, 0.044299107044935226, -0.003433781210333109, -0.0009848899208009243, -0.05895793065428734]


## Task 3: Search Function
`search_chunks(query, chunks, embeddings, k=3)`

Test run showing the top-k most similar chunks for a sample query.

In [5]:
query = "What embedding model is used and why?"

top_results = search_chunks(query, chunks, embeddings, k=3)

print(f"Query: {query}\n")
for i, r in enumerate(top_results):
    print(f"--- Top {i+1} Result ---")
    print(r)
    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3122.68it/s]


Query: What embedding model is used and why?

--- Top 1 Result ---
Popular embedding models include Sentence-Transformers models like all-MiniLM-L6-v2, which is fast and lightweight, making it suitable for local, cost-free embedding generation. For the generation step, commercial APIs such as Cohere's Command models or OpenAI's GPT models are commonly used because

--- Top 2 Result ---
A typical RAG pipeline has three main stages. First, documents are split into smaller chunks so they can be processed efficiently and fit within context limits. Second, each chunk is converted into a numerical vector using an embedding model, and these vectors are stored in a vector database such

--- Top 3 Result ---
vectors are stored in a vector database such as ChromaDB or FAISS. Third, when a user asks a question, the question itself is embedded, and the system searches the vector database for the most similar chunks using a similarity metric like cosine similarity.



In [6]:
# A second query to further validate retrieval quality
query2 = "Why is chunk size and overlap important?"

top_results2 = search_chunks(query2, chunks, embeddings, k=2)

print(f"Query: {query2}\n")
for i, r in enumerate(top_results2):
    print(f"--- Top {i+1} Result ---")
    print(r)
    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6443.15it/s]


Query: Why is chunk size and overlap important?

--- Top 1 Result ---
Chunk size and overlap are important hyperparameters in a RAG system. If chunks are too large, irrelevant information gets mixed in with relevant information, reducing retrieval precision. If chunks are too small, important context might get split across multiple chunks, making it harder for the

--- Top 2 Result ---
across multiple chunks, making it harder for the retriever to find complete answers. Overlap between chunks helps prevent important information from being cut off at chunk boundaries.



In [7]:
from dotenv import load_dotenv
import os

load_dotenv()  # .env file ko load karta hai
COHERE_API_KEY = os.environ.get("COHERE_API_KEY")

## Task 5: Cohere Answer Generation
`generate_answer(query, context, api_key, model_name="command-r-08-2024", temperature=0.2)`

Test run using the retrieved chunks from Task 3 as context.

**Note:** requires a valid Cohere API key. Get a free trial key from
https://dashboard.cohere.com/api-keys

In [8]:
context = "\n\n---\n\n".join(top_results)
answer = generate_answer(query, context, COHERE_API_KEY, model_name="command-r-08-2024", temperature=0.2)

print(f"Query: {query}\n")
print(f"Context used:\n{context}\n")
print(f"Generated Answer:\n{answer}")

Query: What embedding model is used and why?

Context used:
Popular embedding models include Sentence-Transformers models like all-MiniLM-L6-v2, which is fast and lightweight, making it suitable for local, cost-free embedding generation. For the generation step, commercial APIs such as Cohere's Command models or OpenAI's GPT models are commonly used because

---

A typical RAG pipeline has three main stages. First, documents are split into smaller chunks so they can be processed efficiently and fit within context limits. Second, each chunk is converted into a numerical vector using an embedding model, and these vectors are stored in a vector database such

---

vectors are stored in a vector database such as ChromaDB or FAISS. Third, when a user asks a question, the question itself is embedded, and the system searches the vector database for the most similar chunks using a similarity metric like cosine similarity.

Generated Answer:
The embedding model used in this context is Sentence-

## Summary

| Task | Function | Status |
|------|----------|--------|
| Task 1 | `load_and_chunk_document` | ✅ Tested - correctly splits document into chunks |
| Task 2 | `create_embeddings` | ✅ Tested - returns list of 384-dim vectors (MiniLM-L6-v2) |
| Task 3 | `search_chunks` | ✅ Tested - retrieves top-k relevant chunks via cosine similarity |
| Task 5 | `generate_answer` | ✅ Tested - generates grounded answer using Cohere Chat API |

All functions are also integrated together in `app.py`, the Streamlit interface (Task 4).